In [2]:
import torch

torch.manual_seed(0)
d = 64
w_q, w_k, w_v = torch.randn(3, d, d) / d**0.5  # one attention head's projections
tokens = torch.randn(10, d)  # embeddings of a 10-token sequence

# Decode token by token: project only the newest token and append it to the cache
k_cache, v_cache = torch.empty(0, d), torch.empty(0, d)
for x in tokens:
    k_cache = torch.cat([k_cache, (x @ w_k)[None]])
    v_cache = torch.cat([v_cache, (x @ w_v)[None]])
    out_cached = torch.softmax((x @ w_q) @ k_cache.T / d**0.5, dim=-1) @ v_cache

# Without a cache, the last step re-projects keys and values for all 10 tokens
k_full, v_full = tokens @ w_k, tokens @ w_v
out_full = torch.softmax((tokens[-1] @ w_q) @ k_full.T / d**0.5, dim=-1) @ v_full

print(f"Same output: {torch.allclose(out_cached, out_full, atol=1e-5)}")
print(f"Key/value projections over 10 steps: cached {len(tokens)}, recomputed {sum(range(1, len(tokens) + 1))}")

/Users/shawn/Documents/Github/self/deepdive/experiments/kvcache/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Same output: True
Key/value projections over 10 steps: cached 10, recomputed 55


In [14]:
k_cache.shape, v_cache.shape

(torch.Size([10, 64]), torch.Size([10, 64]))